# Proyecto de Grado: Análisis Comparativo del Estado de Resultado Integral
## Metodología CRISP-DM

Este notebook contiene el pipeline de datos para el proyecto, analizando los reportes a la Superintendencia de Sociedades.


## 1. Comprensión de los datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')

# 1. Cargar datos
file_path = './data/310030_Estado de resultado integral, resultado del periodo, por funcion de gasto.xlsx'
df = pd.read_excel(file_path, sheet_name='Sheet1')

# Mostrar dimensiones y tipos
print(f"Dimensiones: {df.shape}")
df.info()

print(f"\nNúmero de empresas únicas (NIT): {df['NIT'].nunique()}")
print("\nDistribución por Fecha de Corte:")
print(df['Fecha de Corte'].value_counts())


## 2. Calidad de datos

In [ ]:
# Duplicados por clave
duplicados = df.duplicated(subset=['NIT', 'Fecha de Corte', 'Periodo']).sum()
print(f"Duplicados por NIT+Fecha+Periodo: {duplicados}")

# Valores faltantes
nulos = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print("\nPorcentaje de Nulos por columna (Top 10):")
print(nulos.head(10))

# Verificación contable (Ingresos - Costo = Ganancia Bruta)
df_calc = df.fillna(0)
calc_bruta = df_calc['Ingresos de actividades ordinarias (Revenue)'] - df_calc['Costo de ventas (CostOfSales)']
diff_bruta = (calc_bruta - df_calc['Ganancia bruta (GrossProfit)']).abs()
print(f"\nDiferencias significativas en Ganancia Bruta (>10): {(diff_bruta > 10).sum()}")

# Verificación Operacional
calc_op = (df_calc['Ganancia bruta (GrossProfit)'] - 
           df_calc['Costos de distribución (DistributionCosts)'] - 
           df_calc['Gastos de administración (AdministrativeExpense)'] - 
           df_calc['Otros gastos (OtherExpenseByFunction)'] + 
           df_calc['Otros ingresos (OtherIncome)'])
diff_op = (calc_op - df_calc['Ganancia (pérdida) por actividades de operación (ProfitLossFromOperatingActivities)']).abs()
print(f"Diferencias significativas en Ganancia Operacional (>10): {(diff_op > 10).sum()}")


## 3. Preparación de datos

In [ ]:
# Limpiar duplicados
df = df.drop_duplicates(subset=['NIT', 'Fecha de Corte', 'Periodo'])

# Crear año
df['Fecha de Corte'] = pd.to_datetime(df['Fecha de Corte'])
df['anio'] = df['Fecha de Corte'].dt.year

# Tratar nulos: las partidas financieras no reportadas se asumen 0
cols_fin = ['Ingresos de actividades ordinarias (Revenue)', 'Costo de ventas (CostOfSales)',
            'Ganancia bruta (GrossProfit)', 'Otros ingresos (OtherIncome)',
            'Costos de distribución (DistributionCosts)', 'Gastos de administración (AdministrativeExpense)',
            'Otros gastos (OtherExpenseByFunction)', 'Otras ganancias (pérdidas) (OtherGainsLosses)',
            'Ganancia (pérdida) por actividades de operación (ProfitLossFromOperatingActivities)',
            'Ganancia (pérdida) (ProfitLoss)']
df[cols_fin] = df[cols_fin].fillna(0)

# Separar vistas
df_actual = df[df['Periodo'] == 'Periodo Actual'].copy()
df_anterior = df[df['Periodo'] == 'Periodo Anterior'].copy()

print(f"Registros Periodo Actual: {len(df_actual)}")
print(f"Registros Periodo Anterior: {len(df_anterior)}")


## 4. Indicadores financieros

In [ ]:
# Margen Bruto, Operacional y Neto (sobre Periodo Actual)
df_actual['margen_bruto'] = np.where(df_actual['Ingresos de actividades ordinarias (Revenue)'] > 0,
                                     df_actual['Ganancia bruta (GrossProfit)'] / df_actual['Ingresos de actividades ordinarias (Revenue)'], np.nan)
df_actual['margen_operacional'] = np.where(df_actual['Ingresos de actividades ordinarias (Revenue)'] > 0,
                                           df_actual['Ganancia (pérdida) por actividades de operación (ProfitLossFromOperatingActivities)'] / df_actual['Ingresos de actividades ordinarias (Revenue)'], np.nan)
df_actual['margen_neto'] = np.where(df_actual['Ingresos de actividades ordinarias (Revenue)'] > 0,
                                    df_actual['Ganancia (pérdida) (ProfitLoss)'] / df_actual['Ingresos de actividades ordinarias (Revenue)'], np.nan)

# Variación respecto a Periodo Anterior
df_merged = pd.merge(df_actual, df_anterior[['NIT', 'Fecha de Corte', 'Ganancia (pérdida) (ProfitLoss)']], 
                     on=['NIT', 'Fecha de Corte'], how='left', suffixes=('', '_Anterior'))

df_merged['var_ganancia_neta'] = np.where(df_merged['Ganancia (pérdida) (ProfitLoss)_Anterior'] != 0,
                                          (df_merged['Ganancia (pérdida) (ProfitLoss)'] - df_merged['Ganancia (pérdida) (ProfitLoss)_Anterior']) / df_merged['Ganancia (pérdida) (ProfitLoss)_Anterior'].abs(),
                                          np.nan)
display(df_merged[['NIT', 'Razón social de la sociedad', 'margen_neto', 'var_ganancia_neta']].head())


## 5. Análisis Exploratorio de Datos (EDA)

In [ ]:
# Separar CIIU para análisis
df_merged[['ciiu_codigo', 'ciiu_sector']] = df_merged['Clasificación Industrial Internacional Uniforme Versión 4 A.C (CIIU)'].str.extract(r'([A-Z0-9]+)\s*-\s*(.*)')
df_merged[['ciiu_codigo', 'ciiu_sector']] = df_merged[['ciiu_codigo', 'ciiu_sector']].fillna('Desconocido')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(df_merged[df_merged['margen_neto'].between(-1, 1)]['margen_neto'], bins=50, ax=axes[0], color='skyblue')
axes[0].set_title('Distribución del Margen Neto (acotado -1 a 1)')

top_sectores = df_merged.groupby('ciiu_sector')['Ganancia (pérdida) (ProfitLoss)'].sum().nlargest(10)
sns.barplot(y=top_sectores.index, x=top_sectores.values, ax=axes[1], palette='viridis')
axes[1].set_title('Top 10 Sectores por Ganancia Neta')
plt.tight_layout()
plt.show()

# Matriz de correlación
vars_corr = ['Ingresos de actividades ordinarias (Revenue)', 'Ganancia bruta (GrossProfit)', 
             'Ganancia (pérdida) por actividades de operación (ProfitLossFromOperatingActivities)', 'Ganancia (pérdida) (ProfitLoss)',
             'margen_neto']
plt.figure(figsize=(8, 6))
sns.heatmap(df_merged[vars_corr].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de Correlación')
plt.show()


## 6. Exportar

In [ ]:
df_merged.to_csv('datos_limpios_dashboard.csv', index=False, encoding='utf-8-sig')
print("Datos limpios guardados en 'datos_limpios_dashboard.csv'")
